# Hydrological Drought Propagation Analysis

This notebook identifies **temporal compatibility** between drought events detected at downstream (origin) stations and events at stations in their upstream drainage network, following the methodology described in `README_propagation.md`.

## Pipeline summary

For each drought event at an origin station:

1. **Retrieve upstream stations** from the connectivity file.
2. **Build an asymmetric search window**: expand the left side by `W_DAYS` = 45 days before origin start to capture precursor droughts; set the right side to origin start (not expanded — the lag filter requires upstream_start ≤ origin_start, making any right-side expansion redundant).
3. **Find candidate upstream events** that intersect the search window *and* have at least `MIN_OVERLAP` = 5 days of **real** (non-expanded) overlap with the origin event.
4. **Select the best event per upstream station** using a deterministic four-level tie-breaking rule (overlap → severity → temporal proximity → event id).
5. **Compute the temporal lag** = upstream start − origin start (days).
6. **Apply lag filter**: retain only pairs where `lag_days ≤ 0` (upstream started before or simultaneously with origin).
7. **Compute chain metrics** per origin event: chain size and propagation fraction.

> **Design note — no chain merging.** Each origin event produces exactly one propagation chain. No post-hoc merging of consecutive or overlapping chains is applied. See the rationale in the *Chain summary* section below.

## Inputs

| File | Description |
|---|---|
| `SSI_drought_events.csv` | Event catalogue produced by `SSI_drought_events.ipynb` |
| `upstream_connectivity.csv` | Per-station list of all hydraulically upstream gauging stations |

## Outputs

| File | Description |
|---|---|
| `data/propagation_W45_minov5.csv` | All compatible pairs (before lag filter) |
| `data/propagation_W45_minov5_lagneg.csv` | Lag ≤ 0 pairs with chain metrics (primary pair-level result) |
| `data/chains_summary_W45_minov5.csv` | **Primary output.** One chain per origin event with aggregate metrics |
| `data/Chains_final.csv` | Alias of `chains_summary` used by downstream analysis scripts |


In [ ]:
# ── Imports and algorithm parameters ──────────────────────────────────────────
import numpy as np
import pandas as pd
from tqdm import tqdm

# ── Parameters ───────────────────────────────────────
W_DAYS      = 45   # days to expand the origin event window in left side
MIN_OVERLAP = 5    # minimum real overlap days for a candidate to qualify
LAG_MAX     = 0    # retain only pairs where lag_days <= this value (safe guard against future leaks)

print(f'Parameters: W_DAYS={W_DAYS}, MIN_OVERLAP={MIN_OVERLAP}, LAG_MAX={LAG_MAX}')

In [ ]:
# ── Load drought events catalogue ─────────────────────────────────────────────
# Produced by SSI_drought_events.ipynb. We parse all date columns upfront.
events = pd.read_csv(
    'data/SSI_drought_events.csv',
    parse_dates=['start_date', 'end_date', 'peak_date']
)
events = events.sort_values(['station_id', 'event_id']).reset_index(drop=True)

print(f'Events loaded : {len(events):,} events across {events["station_id"].nunique()} stations')
print(f'Date range    : {events["start_date"].min().date()} → {events["end_date"].max().date()}')
print()
events.head()

In [ ]:
# ── Load upstream connectivity ─────────────────────────────────────────────────
# File is semicolon-separated; the upstream_chain column is a
# comma-separated string of station IDs (unordered, no topology).
# Rows with an empty upstream_chain represent headwater stations.

conn_raw = pd.read_csv('data/upstream_connectivity.csv', sep=';', dtype=str)
conn_raw.columns = conn_raw.columns.str.strip()
conn_raw['station_id']     = conn_raw['station_id'].str.strip().astype(int)
conn_raw['upstream_chain'] = conn_raw['upstream_chain'].str.strip()

def parse_chain(s):
    """Parse a comma-separated string of station IDs into a list of ints."""
    if pd.isna(s) or s == '':
        return []
    return [int(x.strip()) for x in s.split(',') if x.strip()]

# Build dict: station_id → list of upstream station ids
connectivity = {
    row['station_id']: parse_chain(row['upstream_chain'])
    for _, row in conn_raw.iterrows()
}

n_with_upstream    = sum(1 for v in connectivity.values() if v)
n_without_upstream = len(connectivity) - n_with_upstream

print(f'Stations in connectivity file : {len(connectivity)}')
print(f'  With upstream stations      : {n_with_upstream}')
print(f'  Headwaters (no upstream)    : {n_without_upstream}')
print()

# Preview
for sid, chain in list(connectivity.items())[:5]:
    print(f'  {sid}: {chain}')

## Helper functions

Two focused functions implement the core logic:

- **`real_overlap`** — computes the number of days of actual (non-expanded) overlap between two event intervals. The +1 accounts for inclusive date counting.
- **`select_best_candidate`** — given multiple compatible upstream events at the same station, applies the deterministic four-level tie-breaking rule and returns a single representative event.

In [ ]:
# ── Helper functions ───────────────────────────────────────────────────────────

def real_overlap(s_o, e_o, s_u, e_u):
    """
    Calendar days of real (non-expanded) overlap between two inclusive intervals.
    Returns 0 if the intervals do not overlap.

    Formula: max(0, min(e_o, e_u) - max(s_o, s_u) + 1 day)
    The +1 accounts for inclusive counting (e.g., Jan 1 to Jan 3 = 3 days).
    """
    delta = (min(e_o, e_u) - max(s_o, s_u)).days + 1
    return max(0, delta)


def select_best_candidate(candidates):
    """
    From a DataFrame of compatible upstream events at a single station,
    return the single best row using a deterministic four-level ranking:

      1. overlap_days      — descending (most temporal co-occurrence first)
      2. severity          — descending (more severe event preferred)
      3. abs_start_diff    — ascending  (closest start date to origin preferred)
      4. event_id          — ascending  (lowest id as final tie-break)
    """
    return (
        candidates
        .sort_values(
            ['overlap_days', 'severity', 'abs_start_diff', 'event_id'],
            ascending=[False, False, True, True]
        )
        .iloc[0]
    )


print('Helper functions defined.')

## Main propagation loop

The loop iterates over all origin stations that have at least one upstream station in the connectivity file. For each origin event, it applies the two-stage filter (expanded window → real overlap) and collects one best matched event per upstream station.

**Performance note:** events are pre-indexed by station into a dictionary so each upstream lookup is O(1). The expanded-window filter is vectorised; only the surviving candidates enter the overlap computation.

In [ ]:
# ── Main propagation loop ──────────────────────────────────────────────────────

# Pre-index event table by station for fast lookup
station_events = {
    sid: grp.reset_index(drop=True)
    for sid, grp in events.groupby('station_id')
}

W = pd.Timedelta(days=W_DAYS)

# Only process origin stations that (a) have upstream connectivity and
# (b) have events in the catalogue
origin_stations = [
    sid for sid, chain in connectivity.items()
    if chain and sid in station_events
]

all_pairs = []   # one dict per (origin event, compatible upstream event)

for origin_sid in tqdm(origin_stations, desc='Origin stations'):

    upstream_sids       = connectivity[origin_sid]
    origin_evts         = station_events[origin_sid]
    # Only keep upstream stations present in the event catalogue
    upstream_with_events = [s for s in upstream_sids if s in station_events]

    for _, orig in origin_evts.iterrows():
        s_o = orig['start_date']
        e_o = orig['end_date']

        # Asymmetric search window:
        #   - Left side expanded by W_DAYS to capture precursor upstream droughts
        #     that began weeks before the origin event onset.
        #   - Right side set to origin start date (not expanded) because the lag
        #     filter (lag_days <= 0) requires upstream_start <= origin_start,
        #     making any right-side expansion beyond origin_start redundant.
        w_start = s_o - W
        w_end   = s_o

        for up_sid in upstream_with_events:
            up_evts = station_events[up_sid]

            # ── Stage 1: expanded window filter (vectorised) ───────────────
            mask       = (up_evts['end_date'] >= w_start) & (up_evts['start_date'] <= w_end)
            candidates = up_evts[mask].copy()

            if candidates.empty:
                continue

            # ── Stage 2: real overlap filter ───────────────────────────────
            candidates['overlap_days'] = candidates.apply(
                lambda r: real_overlap(s_o, e_o, r['start_date'], r['end_date']),
                axis=1
            )
            candidates = candidates[candidates['overlap_days'] >= MIN_OVERLAP]

            if candidates.empty:
                continue

            # ── Stage 3: deterministic tie-breaking ────────────────────────
            candidates['abs_start_diff'] = (
                (candidates['start_date'] - s_o).abs().dt.days
            )
            best = select_best_candidate(candidates)

            # ── Stage 4: temporal lag ──────────────────────────────────────
            lag_days = (best['start_date'] - s_o).days

            all_pairs.append({
                'origin_station'   : origin_sid,
                'origin_event_id'  : orig['event_id'],
                'origin_start'     : s_o,
                'origin_end'       : e_o,
                'origin_duration'  : orig['duration'],
                'origin_severity'  : orig['severity'],
                'upstream_station' : up_sid,
                'upstream_event_id': best['event_id'],
                'upstream_start'   : best['start_date'],
                'upstream_end'     : best['end_date'],
                'upstream_duration': best['duration'],
                'overlap_days'     : int(best['overlap_days']),
                'lag_days'         : lag_days,
                'upstream_severity': best['severity'],
            })

# Assemble full results table
prop_full = pd.DataFrame(all_pairs)
prop_full = prop_full.sort_values(
    ['origin_station', 'origin_event_id', 'lag_days', 'upstream_station']
).reset_index(drop=True)

print(f'Compatible pairs (before lag filter) : {len(prop_full):,}')
print(f'Origin events matched to ≥1 upstream : '
      f'{prop_full.groupby(["origin_station","origin_event_id"]).ngroups:,}')
prop_full.head(10)

## Post-processing: lag filter and chain metrics

### Lag filter

Only pairs with `lag_days ≤ 0` are retained. This focuses the analysis on physically plausible propagation patterns where the upstream drought signal **precedes or coincides** with the downstream drought onset. Pairs with positive lags (upstream started after the origin) are excluded.

### Chain metrics

For each origin event two complementary metrics are computed:

| Metric | Definition |
|---|---|
| `chain_size` | Number of upstream stations with a valid (lag ≤ 0) compatible event |
| `n_upstream_with_events` | Total upstream stations present in the event catalogue (denominator) |
| `propagation_fraction` | `chain_size` / `n_upstream_with_events` — normalised [0, 1] |

`propagation_fraction` is necessary because different origin stations have different-sized upstream networks; a raw chain size of 4 is not comparable across basins.

In [ ]:
# ── Lag filter ─────────────────────────────────────────────────────────────────
prop_neg = prop_full[prop_full['lag_days'] <= LAG_MAX].copy().reset_index(drop=True)

print(f'Pairs after lag filter (lag ≤ {LAG_MAX}): {len(prop_neg):,}')
print(f'Pairs excluded (positive lag)           : {len(prop_full) - len(prop_neg):,}')
print()

# ── Chain size per origin event ────────────────────────────────────────────────
# Count upstream stations with at least one valid pair
chain_size = (
    prop_neg
    .groupby(['origin_station', 'origin_event_id'])['upstream_station']
    .nunique()
    .rename('chain_size')
    .reset_index()
)

# ── Denominator: upstream stations present in the event catalogue ──────────────
# Using only stations with actual events avoids penalising origin stations
# whose upstream network includes gauges not present in the dataset.
upstream_available = {
    sid: sum(1 for s in chain if s in station_events)
    for sid, chain in connectivity.items()
}
chain_size['n_upstream_with_events'] = chain_size['origin_station'].map(upstream_available)
chain_size['propagation_fraction']   = (
    chain_size['chain_size'] / chain_size['n_upstream_with_events']
).round(4)

# Merge chain metrics back into the lag-filtered pair table
prop_neg = prop_neg.merge(chain_size, on=['origin_station', 'origin_event_id'], how='left')

print('Chain size summary (across all origin events):')
print(chain_size['chain_size'].describe().round(2))
print()
print('Propagation fraction summary:')
print(chain_size['propagation_fraction'].describe().round(3))
print()
prop_neg.head(10)

In [ ]:
# ── Sanity checks ──────────────────────────────────────────────────────────────

# 1. All retained lags must be <= 0
assert (prop_neg['lag_days'] <= 0).all(), 'Lag filter failed: positive lags present'

# 2. Overlap must be >= MIN_OVERLAP for every pair
assert (prop_neg['overlap_days'] >= MIN_OVERLAP).all(), 'Overlap threshold violated'

# 3. Propagation fraction must be in [0, 1]
assert prop_neg['propagation_fraction'].between(0, 1).all(), 'Propagation fraction out of [0,1]'

# 4. No duplicate (origin_station, origin_event_id, upstream_station) pairs
dups = prop_neg.duplicated(['origin_station', 'origin_event_id', 'upstream_station'])
assert not dups.any(), f'{dups.sum()} duplicate (origin, event, upstream) triples found'

print('All sanity checks passed.')
print()

# ── Overview of lag distribution ───────────────────────────────────────────────
print('Lag distribution (days):')
bins = [(-9999, -90), (-90, -30), (-30, -7), (-7, 0)]
for lo, hi in bins:
    n = ((prop_neg['lag_days'] > lo) & (prop_neg['lag_days'] <= hi)).sum()
    pct = n / len(prop_neg) * 100
    label = f'({lo}, {hi}]' if lo != -9999 else f'< {hi}'
    print(f'  {label:>15} days: {n:>5} pairs ({pct:.1f}%)')

print()
print(f'Median lag : {prop_neg["lag_days"].median():.0f} days')
print(f'Mean lag   : {prop_neg["lag_days"].mean():.1f} days')

In [ ]:
# ── Save outputs ───────────────────────────────────────────────────────────────

# Full table (before lag filter): useful for sensitivity analyses
out_full = f'data/propagation_W{W_DAYS}_minov{MIN_OVERLAP}.csv'
prop_full.to_csv(out_full, index=False)
print(f'Saved {len(prop_full):,} pairs → {out_full}')

# Lag-filtered table (primary result)
out_neg  = f'data/propagation_W{W_DAYS}_minov{MIN_OVERLAP}_lagneg.csv'
prop_neg.to_csv(out_neg, index=False)
print(f'Saved {len(prop_neg):,} pairs → {out_neg}')

print()
print('=== Final summary ===')
print(f'Origin stations processed              : {len(origin_stations)}')
print(f'Total origin events                    : '
      f'{sum(len(station_events[s]) for s in origin_stations):,}')
print(f'Compatible pairs (lag ≤ {LAG_MAX})         : {len(prop_neg):,}')
print(f'Origin events with ≥1 upstream match   : '
      f'{prop_neg.groupby(["origin_station","origin_event_id"]).ngroups:,}')
print(f'Mean chain size                        : '
      f'{chain_size["chain_size"].mean():.2f}')
print(f'Mean propagation fraction              : '
      f'{chain_size["propagation_fraction"].mean():.3f}')

## Chain summary: one chain per origin event

Each unique `(origin_station, origin_event_id)` pair produces exactly one propagation chain. **No post-hoc merging of consecutive chains is applied.**

### Scientific rationale for removing the merge step

An earlier version of this notebook merged chains whose `[chain_start, chain_end]` intervals overlapped, with the intention of consolidating the same drought episode captured by multiple consecutive origin events. This step has been removed for two reasons:

**1. Merging operates at the wrong level.** Chain intervals overlap because of long-duration *upstream* events, not because of continuity at the *origin* station. Two origin drought events that are months apart (e.g. April and November of the same year) can produce overlapping chain windows simply because a single upstream gauge remained in continuous drought throughout. Merging them creates a synthetic, multi-month episode that has no physical counterpart in the origin station record.

**2. Loss of origin event integrity.** After merging, the `origin_end` field is set to the end of the *last* sub-chain's origin event. This means the merged 'origin event' encompasses long recovery periods above the SSI threshold, directly contradicting the event definition used in `SSI_drought_events.ipynb`.

The drought event catalogue already guarantees non-overlapping, continuous events at each station. There is therefore no true double-counting: each origin event is a distinct, well-defined, physically consistent observation. Computing all metrics per origin event is the most transparent and reproducible approach for peer-reviewed publication.


In [ ]:
# ── Chain summary: PRIMARY analytical output (one chain per origin event) ──────

# Add chain_id to prop_neg (idempotent)
prop_neg['chain_id'] = (
    prop_neg['origin_station'].astype(str) + '_' +
    prop_neg['origin_event_id'].astype(str)
)

# Fetch severity_hm3 lookup from events catalogue
hm3 = events[['station_id', 'event_id', 'severity_hm3']]

# Temporary join for upstream hm3 — does NOT modify prop_neg
prop_hm3 = prop_neg.merge(
    hm3.rename(columns={'station_id'  : 'upstream_station',
                        'event_id'    : 'upstream_event_id',
                        'severity_hm3': 'upstream_severity_hm3'}),
    on=['upstream_station', 'upstream_event_id'], how='left'
)

# Origin-level fields: one row per chain
origin_fields = (
    prop_neg.drop_duplicates('chain_id')
    [['chain_id', 'origin_station', 'origin_event_id',
      'origin_start', 'origin_end', 'origin_severity',
      'chain_size', 'n_upstream_with_events', 'propagation_fraction']]
    .merge(
        hm3.rename(columns={'station_id'  : 'origin_station',
                            'event_id'    : 'origin_event_id',
                            'severity_hm3': 'origin_severity_hm3'}),
        on=['origin_station', 'origin_event_id'], how='left'
    )
)

# Aggregate upstream metrics per chain
upstream_agg = (
    prop_hm3.groupby('chain_id')
    .agg(
        chain_start       = ('upstream_start',        'min'),
        upstream_end_max  = ('upstream_end',           'max'),
        lag_mean          = ('lag_days',               'mean'),
        upstream_sev_sum  = ('upstream_severity',      'sum'),
        upstream_hm3_sum  = ('upstream_severity_hm3',  'sum'),
    )
    .reset_index()
)

# Assemble
chains = origin_fields.merge(upstream_agg, on='chain_id')

# chain_end: latest date across upstream AND origin
chains['chain_end']      = chains[['upstream_end_max', 'origin_end']].max(axis=1)
chains['chain_duration'] = (chains['chain_end'] - chains['chain_start']).dt.days + 1

# chain_size: include origin station
chains['chain_size'] = chains['chain_size'] + 1

# chain_severity: sum over all events (upstream + origin)
chains['chain_severity']     = (chains['upstream_sev_sum'] + chains['origin_severity']).round(4)
chains['chain_severity_hm3'] = (chains['upstream_hm3_sum'] + chains['origin_severity_hm3']).round(4)
chains['lag_mean']           = chains['lag_mean'].round(1)

# Final column selection and order
chains = (
    chains[[
        'chain_id', 'origin_station',
        'origin_start', 'origin_end',
        'chain_start', 'chain_end', 'chain_duration',
        'chain_size', 'propagation_fraction',
        'lag_mean',
        'chain_severity', 'chain_severity_hm3',
    ]]
    .sort_values(['origin_station', 'chain_start'])
    .reset_index(drop=True)
)

# Save
# Save: versioned name + alias for downstream scripts
chains.to_csv(f'data/chains_summary_W{W_DAYS}_minov{MIN_OVERLAP}.csv', index=False)
chains.to_csv('data/Chains_final.csv', index=False)  # alias used by plot/table scripts

print(f'Chain summary: {len(chains):,} chains across {chains["origin_station"].nunique()} origin stations')
print()
chains.head(10)

In [ ]:
# Primary output summary statistics
print(f'Primary output: {len(chains):,} chains (one per origin event)')
chains.describe()
